# Eval-determinism gate — **robosuite / LIBERO**, the stack that actually matters

The gymnasium version of this test ran 2026-09-12 19:32 UTC in the Composio workbench and returned
**HARNESS SOUND**: on `Reacher-v5`, stepping does not advance the env PRNG, and a 6-reset run and a
5-reset run share the same first five episodes. **The predicted failure did not happen.**

That result does **not** transfer to LIBERO, for three reasons found by reading LIBERO's source:

1. **LIBERO does not use gymnasium's `reset(seed=)`.** `ControlEnv.seed(seed)` calls
   `self.env.seed(seed)` — robosuite's own method — and `reset()` takes no seed at all.
2. **robosuite 1.4's placement samplers draw from the GLOBAL `np.random`**, not a per-env PRNG.
   A global RNG can be advanced by *any other code in the process*; a per-env PRNG cannot. That is
   a different failure mode from the one already refuted, and it is what **T5** below probes.
3. 🔴 **`ControlEnv.reset()` retries on randomization failure**, verbatim from
   `libero/libero/envs/env_wrapper.py`:

   ```python
   def reset(self):
       success = False
       while not success:
           try:
               ret = self.env.reset(); success = True
           except RandomizationError:
               pass
           finally:
               continue
       return ret
   ```

   **Each retry consumes more randomness.** So the number of RNG draws per reset is *variable* and
   depends on how many times placement sampling failed. Two arms can therefore consume different
   amounts of randomness per episode with no difference in step count at all. This is a concrete
   mechanism the gymnasium test could not have exposed.

## Pre-registered predictions — written before the run, do not edit afterwards

| Test | Question | **Prediction** |
|---|---|---|
| **T1** | `seed(s)` then `reset()` twice, fresh envs → identical sim state? | **PASS** |
| **T2** | reset after K steps, no reseed → does the next episode depend on K? | **PASS** (i.e. no step-count effect, matching the gymnasium result) |
| **T3** | same, but `seed()` called again before each reset | **PASS** |
| **T4** | 6 consecutive resets vs 5 — is the 5-run a prefix of the 6-run? | **PASS** (aligned) |
| **T5** | **consume 1000 draws from global `np.random` between resets** → does the next episode change? | 🔴 **FAIL — the state WILL differ** |

**T5 is the one that matters.** If it fails, then in LIBERO any code that touches the global numpy
RNG between episodes — including a policy that samples actions — shifts every subsequent episode's
initial state, and two arms that draw different amounts are scored on different episodes. **That is
the paper.** If T5 passes, P6 is dead and should be recorded as dead.

**Verdict rule, fixed in advance:** `CONFOUND CONFIRMED` iff T1 passes and (T2 fails **or** T5
fails). `HARNESS SOUND` iff T1, T2 and T5 all pass. `INCONCLUSIVE` if any control disagrees with
itself. `BROKEN` if T1 fails.


## Cell 1 — install LIBERO's own pinned stack

**Version notes, because the result is meaningless without them:**

- LIBERO's `requirements.txt` pins `robosuite==1.4.0`, `numpy==1.22.4`, `gym==0.25.2`,
  `bddl==1.0.1`. `robosuite==1.4.0` is required, not cosmetic: `ControlEnv` calls
  `suite.load_controller_config(...)`, which robosuite 1.5 removed.
- **`numpy==1.22.4` has no wheel for Colab's Python.** `1.26.4` is the newest `numpy<2` that does,
  so it is used instead. This is a deliberate deviation from LIBERO's pin and it is recorded in the
  output JSON.
- **mujoco is left to pip's resolver rather than pinned to 3.3.3.** robosuite 1.4.0 predates
  mujoco 3.x, and forcing the pin may simply fail to import. If it resolves to a 2.x, that is
  itself a finding worth reporting, and it is why the JSON records the resolved version. Pin it
  afterwards only if you want to match LIBERO issue #88 exactly.

Expect this cell to take several minutes and to print dependency-resolver warnings. Read the
`RESOLVED` block at the end, not the warnings.


In [ ]:
import subprocess, sys

def pip(*a):
    print(">>> pip", *a, flush=True)
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a]).returncode

pip("numpy==1.26.4")
pip("robosuite==1.4.0", "bddl==1.0.1", "gym==0.25.2", "easydict", "opencv-python")
pip("git+https://github.com/Lifelong-Robot-Learning/LIBERO.git", "--no-deps")

import importlib
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym"):
    try:
        mod = importlib.import_module(m)
        print(f"RESOLVED {m:12s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"RESOLVED {m:12s} IMPORT FAILED: {type(e).__name__}: {e}")

## Cell 2 — build the env on one LIBERO-Object task

`get_sim_state()` is LIBERO's own accessor (`self.env.sim.get_state().flatten()`), so the
fingerprint is the full MuJoCo state vector: no rendering, no policy, no checkpoint.

`MUJOCO_GL=egl` is set before the first import because Colab is headless and `OffScreenRenderEnv`
forces `has_offscreen_renderer=True`. If EGL fails, try `osmesa`.


In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")

import hashlib, json, datetime
import numpy as np

TASK_SUITE = "libero_object"
TASK_NAME  = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
SEEDS      = [11, 12, 13]
K_SHORT, K_LONG = 40, 80
GLOBAL_DRAWS = 1000          # how many np.random.random() calls T5 burns between resets

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

_b = benchmark.get_benchmark_dict()[TASK_SUITE]()
_names = [_b.get_task(i).name for i in range(_b.n_tasks)]
TASK_ID = _names.index(TASK_NAME)
_task = _b.get_task(TASK_ID)
BDDL = os.path.join(get_libero_path("bddl_files"), _task.problem_folder, _task.bddl_file)
print("task", TASK_ID, _task.name)
print("bddl", BDDL)

def mk():
    return OffScreenRenderEnv(bddl_file_name=BDDL, camera_heights=128, camera_widths=128)

def key(env):
    s = np.asarray(env.get_sim_state(), dtype=np.float64)
    return hashlib.sha256(np.ascontiguousarray(s).tobytes()).hexdigest()[:16]

def acts(k, tag, env):
    rs = np.random.RandomState(1234 if tag == "armA" else 5678)
    dim = env.env.action_dim
    return [rs.uniform(-0.2, 0.2, size=dim) for _ in range(k)]

R = {"task": TASK_NAME, "seeds": SEEDS, "k_short": K_SHORT, "k_long": K_LONG,
     "global_draws": GLOBAL_DRAWS,
     "numpy_pin_deviation": "LIBERO pins numpy==1.22.4; used 1.26.4 (newest numpy<2 with a wheel)"}
print("ready")

## Cell 3 — T1 · is a seeded reset reproducible at all?

Mirrors exactly what `LiberoEnv.reset()` does: `env.seed(int(seed))` then `env.reset()`.
If this fails, nothing below is interpretable and the problem is the install.


In [ ]:
def t1_once(s):
    e = mk(); e.seed(int(s)); e.reset(); k = key(e); e.close(); return k

R["T1"] = {}
for s in SEEDS:
    a, b = t1_once(s), t1_once(s)
    R["T1"][s] = {"a": a, "b": b, "match": a == b}
    print(f"seed {s}: {a} vs {b} -> {'MATCH' if a==b else 'DIFFER'}")

T1_PASS = all(v["match"] for v in R["T1"].values())
print("\nT1:", "PASS" if T1_PASS else "FAIL")

## Cell 4 — T2 · does the next episode depend on STEPS taken?

In [ ]:
def roll_reset(s, k, tag, reseed=None, burn=0):
    e = mk(); e.seed(int(s)); e.reset()
    for a in acts(k, tag, e):
        e.step(a)
    for _ in range(burn):
        np.random.random()                 # T5 only: advance the GLOBAL numpy RNG
    if reseed is not None:
        e.seed(int(reseed))
    e.reset()                              # unseeded unless reseed given
    kk = key(e); e.close(); return kk

R["T2"] = {}
for s in SEEDS:
    sh = roll_reset(s, K_SHORT, "armA")
    lo = roll_reset(s, K_LONG,  "armB")
    ct = roll_reset(s, K_SHORT, "armA")
    R["T2"][s] = {"short": sh, "long": lo, "control": ct,
                  "arms_match": sh == lo, "control_match": sh == ct}
    print(f"seed {s}: K{K_SHORT}={sh} K{K_LONG}={lo} arms="
          f"{'MATCH' if sh==lo else 'DIFFER'} ctrl={'ok' if sh==ct else 'UNSTABLE'}")

CONTROL_OK = all(v["control_match"] for v in R["T2"].values())
T2_PASS    = all(v["arms_match"]    for v in R["T2"].values())
print("\ncontrol:", "ok" if CONTROL_OK else "UNSTABLE - T2 uninterpretable")
print("T2:", "PASS (no step-count effect)" if T2_PASS else "FAIL (state depends on step count)")

## Cell 5 — T3 · does calling `seed()` again before each reset fix it?

In [ ]:
R["T3"] = {}
for s in SEEDS:
    a = roll_reset(s, K_SHORT, "armA", reseed=s + 1)
    b = roll_reset(s, K_LONG,  "armB", reseed=s + 1)
    R["T3"][s] = {"a": a, "b": b, "match": a == b}
    print(f"seed {s}: {a} vs {b} -> {'MATCH' if a==b else 'DIFFER'}")

T3_PASS = all(v["match"] for v in R["T3"].values())
print("\nT3:", "PASS - explicit reseeding makes episodes arm-independent"
      if T3_PASS else "FAIL - reseeding does not fix it")

## Cell 6 — T4 · the mhh-gate 6-vs-5 shape

The 2026-08-25 check compared a 6-episode run against a 5-episode run and called it
`IDENTICAL:False`. This asks the fair version: are the five shared episodes the *same* five?


In [ ]:
def seq(s, n):
    e = mk(); e.seed(int(s)); e.reset(); out = [key(e)]
    for _ in range(n - 1):
        e.reset(); out.append(key(e))
    e.close(); return out

R["T4"] = {}
for s in SEEDS:
    six, five = seq(s, 6), seq(s, 5)
    R["T4"][s] = {"six": six, "five": five,
                  "prefix_aligned": six[:5] == five,
                  "all_distinct": len(set(six)) == len(six)}
    print(f"seed {s}: prefix_aligned={six[:5]==five} all_distinct={len(set(six))==len(six)}")

T4_ALIGNED = all(v["prefix_aligned"] for v in R["T4"].values())
print("\nT4:", "aligned - unequal episode counts share the same prefix"
      if T4_ALIGNED else "NOT aligned - the two runs diverge outright")

## Cell 7 — 🔴 T5 · the global-RNG probe, the test this notebook exists for

Identical seed, identical step count, identical everything — except one arm burns
`GLOBAL_DRAWS` calls of `np.random.random()` before its reset.

**If the next episode differs, robosuite's placement sampling is reading the global numpy RNG**,
and in LIBERO any code that touches `np.random` between episodes (a policy sampling actions, a
logger, an augmentation) silently shifts every later episode. That is an arm-dependent confound
with no step-count component, and the gymnasium test could not have found it.


In [ ]:
R["T5"] = {}
for s in SEEDS:
    base = roll_reset(s, K_SHORT, "armA", burn=0)
    burn = roll_reset(s, K_SHORT, "armA", burn=GLOBAL_DRAWS)
    ctrl = roll_reset(s, K_SHORT, "armA", burn=0)
    R["T5"][s] = {"no_burn": base, "burned": burn, "control": ctrl,
                  "match": base == burn, "control_match": base == ctrl}
    print(f"seed {s}: no_burn={base} burned={burn} -> "
          f"{'MATCH (global RNG not used)' if base==burn else 'DIFFER (GLOBAL RNG IS USED)'}"
          f" | ctrl={'ok' if base==ctrl else 'UNSTABLE'}")

T5_CONTROL_OK = all(v["control_match"] for v in R["T5"].values())
T5_PASS       = all(v["match"]         for v in R["T5"].values())
print("\nT5:", "PASS - global RNG does not affect episode sequence"
      if T5_PASS else "FAIL - the global numpy RNG DOES shift the episode sequence")

## Cell 8 — verdict

In [ ]:
import importlib
ver = {}
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym"):
    try:
        ver[m] = getattr(importlib.import_module(m), "__version__", "?")
    except Exception as e:
        ver[m] = f"IMPORT FAILED: {type(e).__name__}"

ts = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

if not (CONTROL_OK and T5_CONTROL_OK):
    verdict = "INCONCLUSIVE"
    reading = "A control disagreed with itself; the harness is noisy for a reason this test does not isolate."
elif not T1_PASS:
    verdict = "BROKEN"
    reading = "Seeded resets are not reproducible at all; suspect the install before LIBERO."
elif T2_PASS and T5_PASS:
    verdict = "HARNESS SOUND"
    reading = ("Neither step count nor global-RNG consumption changes the episode sequence. "
               "P6 is dead: record it as dead and do not revive it without a new mechanism.")
else:
    verdict = "CONFOUND CONFIRMED"
    which = []
    if not T2_PASS: which.append("step count")
    if not T5_PASS: which.append("global-RNG consumption")
    reading = ("The episode sequence depends on " + " and ".join(which) +
               ", so two arms differing in that respect are scored on different episodes. "
               "This is the paper.")

R.update({"timestamp_utc": ts, "versions": ver, "verdict": verdict, "reading": reading,
          "T1_pass": T1_PASS, "T2_pass": T2_PASS, "T3_pass": T3_PASS,
          "T4_prefix_aligned": T4_ALIGNED, "T5_pass": T5_PASS,
          "control_ok": CONTROL_OK, "t5_control_ok": T5_CONTROL_OK,
          "predictions": {"T1": "PASS", "T2": "PASS", "T3": "PASS",
                          "T4": "PASS", "T5": "FAIL (pre-registered)"}})

with open("/content/det_result_robosuite.json", "w") as f:
    json.dump(R, f, indent=2, default=str)

print(json.dumps({k: R[k] for k in
                  ["timestamp_utc", "verdict", "reading", "versions", "T1_pass", "T2_pass",
                   "T3_pass", "T4_prefix_aligned", "T5_pass", "control_ok", "t5_control_ok",
                   "predictions"]}, indent=2))
print("\nfull record written to /content/det_result_robosuite.json")
print("PASTE THE BLOCK ABOVE BACK for Paper Choice 2026-09-12.md section 15")